# Causal Graph Fuzzy (MISO)

In [1]:
from AUTODCETS import datasets
import pandas as pd
import numpy as np
import sys
sys.path.append('CGF-LLM')
import CGFuzzy, utils

2025-07-14 15:55:45,427	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


In [15]:
name_dataset = 'DEC'
target = 'active_power'
df = utils.get_dataset(name = name_dataset)

max_lags = 20
partitions = 5
database_path = 'CGFuzzy_tri_5.db'
# epochs = 20
model_name = 'CGFuzzy'

windows = utils.rolling_window(df, 10)
utils.execute("CREATE TABLE IF NOT EXISTS results(model TEXT, name_dataset TEXT, window INT, predict FLOAT, \
               real FLOAT)", database_path)

[]

In [16]:
# Loop through the windows
for i, window in enumerate(windows):
    if i < 0:
      print('window already executed')
    else:
      # Exclude constant series.
      for variable in window.columns:
          if window[variable].max() == window[variable].min():
              window = window.drop(variable, axis=1)
              print(f"Variables {variable} were deleted because they are constant.")

      idx = int(0.8 * len(window))
      train_dataset, test_dataset = window.head(idx), window.tail(len(window) - idx)

      # CGFuzzy
      ts_fuzzy, partitioners = CGFuzzy.fuzzyfy(train_dataset, npart=partitions)

      model, graph = CGFuzzy.fit(train_dataset, ts_fuzzy, target, max_lags)

      forecasts = CGFuzzy.predict(test_dataset, model, graph, max_lags, target, partitioners)
      real = test_dataset[target].squeeze().tolist()[max_lags:]

      # Cria lista de tuplas com os dados
      rows_to_insert = [
          (model_name, name_dataset, i, forecast, real_val)
          for forecast, real_val in zip(forecasts, real)
      ]

      # Insere tudo de uma vez
      utils.executemany(
          "INSERT INTO results VALUES(?, ?, ?, ?, ?)",
          rows_to_insert,
          database_path
      )
      # for j in range(len(forecasts)):
      #     utils.execute_insert("INSERT INTO results VALUES(?, ?, ?, ?, ?)", (model_name, name_dataset, i, forecasts[j], real[j]), database_path)

      print(f'Save window: {i}')

Save window: 0
Save window: 1
Save window: 2
Save window: 3
Save window: 4
Save window: 5
Save window: 6
Save window: 7
Save window: 8
Save window: 9


## How to view the results

In [17]:
results = utils.get_metrics(database_path)
results

c:\Users\Patricia\OneDrive\Área de Trabalho\FUZZY_CAUSAL_LLM\IEEE\flautim\.venv\lib\site-packages\numpy\_core\fromnumeric.py:3596: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
c:\Users\Patricia\OneDrive\Área de Trabalho\FUZZY_CAUSAL_LLM\IEEE\flautim\.venv\lib\site-packages\numpy\_core\_methods.py:140: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret / rcount


[     Model  Dataset  AVG NRMSE  STD NRMSE
 0  CGFuzzy  BITCOIN   0.901738   0.528111,
      Model Dataset  AVG NRMSE  STD NRMSE
 0  CGFuzzy   SONDA   0.286761   0.062056,
      Model Dataset  AVG NRMSE  STD NRMSE
 0  CGFuzzy     DEC    0.46774   0.207439]